# Data Visualization

Numbers in a table are hard to compare and summary statistics can hide the shape of the data. A good chart, or visualization, does two different jobs. It helps *us* (the data analysts) understand the data, and it helps *someone else* understand what we found. This notebook covers the chart types we use most in this course (bar charts, histograms, boxplots, line charts, and scatter plots), how the `seaborn` plotting functions are organized, and some best practices for good visualizations.

Before starting, work through the [Working with Data](../04_working_data) module. This module assumes you are comfortable loading, filtering, reshaping, and summarizing a dataset with `pandas`.

You have already made several of these charts in earlier notebooks: a line chart of the coozie experiment in [Python Basics](../02_python_basics/python_basics.ipynb), and histograms, bar charts, and a scatter plot of the diamonds data in [Statistical Concepts](../03_data_analytics_statistics/statistical_concepts.ipynb). Here, we slow down a little and look at each chart type on its own, then step back and talk about what makes a chart good.

## General housekeeping items
As in the previous notebooks, select your kernel before running anything. Click **Select Kernel** in the top right of this notebook, choose **Python Environments...**, and select the `.venv` environment you created in [Getting Started](../01_getting_started/course_intro.ipynb).

Let's begin by importing the necessary libraries. We will use `pandas` for data manipulation, `numpy` for numerical operations, and `seaborn` (with `matplotlib`) for data visualization. These are the same four packages you imported in [Python Basics](../02_python_basics/python_basics.ipynb). We add one new import, `matplotlib.ticker`, which we use to format axis tick labels (e.g., adding commas or a dollar sign).

We also do one new thing in this notebook: set a `seaborn` **theme**. A theme is a set of default colors, fonts, and grid lines that applies to every plot you draw after you set it. The earlier notebooks used the plain `matplotlib` defaults. The `whitegrid` style adds light grid lines, which make it easier to read values off a chart. When we set a theme in this course, this is the one we use. `seaborn` has five built-in styles (`darkgrid`, `whitegrid`, `dark`, `white`, and `ticks`), and the theme also controls the color palette and font sizes. The [seaborn aesthetics tutorial](https://seaborn.pydata.org/tutorial/aesthetics.html) and [`set_theme()` documentation](https://seaborn.pydata.org/generated/seaborn.set_theme.html) contain additional information on this feature. 

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick # Format axis tick labels (e.g., add commas)

sns.set_theme(style="whitegrid") # Apply the same look to every plot in this notebook

## Import datasets

In this notebook we work with five datasets. All of them are stored in this folder alongside the notebook, and the [README](README.md) describes each one.

The GDP data is our first Excel file. `pd.read_excel()` works just like `pd.read_csv()`. Let's import it and take a look.

In [ ]:
gdp = pd.read_excel('imf_weo_data.xlsx') # Import the GDP data from an Excel file

gdp.head()

A few wrangling things here. Notice the column headers. The year columns (`2022`, `2023`, ...) are numbers rather than text, which makes them awkward to refer to. We will convert all of the column names to text first.

Then we will keep the ten largest economies in 2022. The spreadsheet ends with a few footnote rows that have no GDP value. Sorting puts those rows last, so they fall out when we keep the top ten.

In [ ]:
gdp.columns = gdp.columns.astype(str) # Year headers were read as numbers; make them text

gdp_top10 = (
    gdp
    .sort_values('2022', ascending=False) # Largest 2022 GDP first
    .head(10) # Keep the ten largest economies
    .reset_index(drop=True) # Renumber the rows 0 to 9
)

gdp_top10[['Country', '2022']]

Now let's import the other four datasets (all pickle files). The diamonds and Titanic data come from earlier modules. As in [Statistical Concepts](../03_data_analytics_statistics/statistical_concepts.ipynb), we keep diamonds under 1.5 carats in a variable called `diamonds_small`.

In [ ]:
diamonds = pd.read_pickle('diamonds.pkl')
diamonds_small = diamonds[diamonds['carat'] < 1.5] # Keep only diamonds under 1.5 carats

titanic = pd.read_pickle('titanic.pkl')

taxis = pd.read_pickle('taxis.pkl')

ua_ut_seasons = pd.read_pickle('ua_ut_seasons.pkl')

The taxi and football datasets are new, so let's inspect them before plotting anything. Remember the advice from [Working with Data](../04_working_data): make sure you understand the data before you analyze it.

In [ ]:
taxis.head()

In [ ]:
taxis.info()

Each row is one taxi trip, with pickup and dropoff times (already stored as dates), the fare, tip, and total, the payment type, and the pickup and dropoff locations. A handful of trips are missing data (e.g., payment type or pickup borough).

In [ ]:
ua_ut_seasons.head()

Each row is one season for one school. The `value` column is the share of games the school won that season (so 0.733 means the team won about 73% of its games). Notice that this data is already tidy and in long format, with one row per school and year, which is the shape that `seaborn` likes best.

## Bar charts

Bar charts compare a value (a count, a sum, or an average) across **categories**. The **length** of the bar is the value, which is why bar charts should generally start at zero... more on that later. We use `sns.barplot()`, the same function we used in [Statistical Concepts](../03_data_analytics_statistics/statistical_concepts.ipynb) to plot average price by clarity. Let's start with 2022 GDP for the ten largest economies.

In [ ]:
sns.barplot(data=gdp_top10,
            x='Country',
            y='2022'
            )

plt.show() # Display the plot

That is a complete chart with one function call, and it already tells us something. But look at the labels. `seaborn` reused the column names, so the y-axis reads `2022` and there is no title at all. A reader who did not write the code has no idea what `2022` is measuring or in what units.

This is where `matplotlib` comes in. `seaborn` draws the chart, and then we use a few `plt` functions to finish it before `plt.show()` displays it:

- `plt.title()` adds a title above the chart.
- `plt.xlabel()` and `plt.ylabel()` replace the axis labels.

In this course, we want every chart to have a title, and we replace the axis labels whenever the column names do not already describe the axis completely. Let's add them.

In [ ]:
sns.barplot(data=gdp_top10,
            x='Country',
            y='2022'
            )

plt.title('GDP by Country (2022)') # Add a title to the plot
plt.xlabel('Country') # Add x-axis label
plt.ylabel('GDP (Billions of USD)') # Add y-axis label

plt.show()

The country names on the x-axis run into each other. When category labels are long, a good fix is to flip the axes: put the categories on the y-axis so the bars run horizontally and the labels have room to breathe. All we do is swap `x` and `y` (and the titles).

In [ ]:
sns.barplot(data=gdp_top10,
            x='2022',
            y='Country'
            )

plt.title('GDP by Country (2022)')
plt.xlabel('GDP (Billions of USD)')
plt.ylabel('Country')

plt.show()

Much easier to read. Now, let's make it presentation ready and save the file. We will make the figure wider with `plt.figure(figsize=(width, height))`, which has to come *before* the plot call, and we will add thousands separators to the tick labels with `matplotlib.ticker`. The bars are already in descending order because we sorted the data when we created `gdp_top10`.

In [ ]:
plt.figure(figsize=(8, 4)) # Set the figure size in inches before plotting

sns.barplot(data=gdp_top10,
            x='2022',
            y='Country'
            )

plt.title('The Ten Largest Economies in 2022', fontsize=14, weight='bold')
plt.xlabel('GDP (Billions of Current USD)', fontsize=12)
plt.ylabel('') # The country names speak for themselves
plt.gca().xaxis.set_major_formatter(mtick.StrMethodFormatter('{x:,.0f}')) # Add commas to the tick labels

plt.savefig('gdp_top10.png', dpi=300, bbox_inches='tight') # Save the figure before displaying it

plt.show()

## Histograms

Histograms show the distribution of a single numeric variable by dividing its range into bins and counting the observations in each one. We used `sns.histplot()` in [Statistical Concepts](../03_data_analytics_statistics/statistical_concepts.ipynb) to look at diamond prices. The `bins` argument controls how many bins there are. Too few and you smooth away the shape, too many and you see noise. Generally speaking, the more observations you have, the more bins you can use. Let's start with diamond prices again.

In [ ]:
sns.histplot(data=diamonds_small,
             x='price',
             bins=30
             )

plt.title('Distribution of Diamond Prices')
plt.xlabel('Price (USD)')
plt.ylabel('Count')

plt.show()

Change `bins` to 10 and then to 100 and rerun the cell. Notice how the impression of the shape changes even though the data does not. There is no single correct number of bins, so try a few.

Now let's look at taxi fares. Compare this shape to the diamond prices above.

In [ ]:
sns.histplot(data=taxis,
             x='fare',
             bins=30
             )

plt.title('Distribution of Taxi Fares')
plt.xlabel('Fare (USD)')
plt.ylabel('Count')

plt.show()

Both are positively skewed, with a long right tail. Most fares are small, and a few are very large.

Finally, let's plot the ages of passengers on the Titanic. About 177 passengers have no recorded age. `seaborn` drops missing values before plotting, so the chart shows only the passengers with an age.

In [ ]:
sns.histplot(data=titanic,
             x='age',
             bins=30
             )

plt.title('Age of Passengers on the Titanic')
plt.xlabel('Age (Years)')
plt.ylabel('Count')

plt.show()

## Boxplots

A boxplot summarizes a distribution with the statistics we covered in [Statistical Concepts](../03_data_analytics_statistics/statistical_concepts.ipynb). Boxplots are an *exploring* tool more than a *presenting* tool. Most audiences outside of a statistics class have never been taught to read one. 

The line inside the box is the median, the edges of the box are the 25th and 75th percentiles (so the box spans the interquartile range), the whiskers extend to the most extreme values within 1.5 times the IQR, and anything beyond the whiskers is drawn as an individual point. **Look at the following graph for an [instructive illustration](https://miro.medium.com/max/9000/1*2c21SkzJMf3frPXPAR_gZA.png).** 

Because they are so compact, boxplots are great for comparing the distribution of a variable across several groups at once. Let's look at passenger age by class on the Titanic.

In [ ]:
sns.boxplot(data=titanic,
            x='class',
            y='age'
            )

plt.title('Passenger Age by Class on the Titanic')
plt.xlabel('Passenger Class')
plt.ylabel('Age (Years)')

plt.show()

First class passengers tend to be older, and the spread of ages varies across classes. Now let's compare taxi fares by pickup borough.

In [ ]:
sns.boxplot(data=taxis,
            x='fare',
            y='pickup_borough'
            )

plt.title('Taxi Fare by Pickup Borough')
plt.xlabel('Fare (USD)')
plt.ylabel('Pickup Borough')

plt.show()

Notice the long upper whiskers and the many points above them. This is the same right skew we saw in the histogram of fares, now visible within each borough.

## Line charts

Line charts show how a value changes across an *ordered* variable, which almost always means time. The line connects consecutive points, so it implies that the value moves continuously from one point to the next. That is reasonable for years or months, and misleading for categories (we will see an example of this at the end of the notebook). We used `sns.lineplot()` in [Python Basics](../02_python_basics/python_basics.ipynb) to plot the coozie experiment. Here, let's plot the season records of Alabama and Tennessee since 1996, using `hue` to draw one line per school.

In [ ]:
sns.lineplot(data=ua_ut_seasons,
             x='year',
             y='value',
             hue='school_record'
             )

plt.title('Football Season Records')
plt.xlabel('Season')
plt.ylabel('Share of Games Won')
plt.legend(title='School') # Add legend title

plt.show()

Let's make this look better. We can pass a dictionary to `palette` to assign a specific color to each school (the official school colors, naturally). We will also add markers so each season is visible, format the y-axis as percentages, and widen the figure.

In [ ]:
school_colors = {'Alabama': '#9E1B32', 'Tennessee': '#FF8200'} # Official school colors

plt.figure(figsize=(9, 4))

sns.lineplot(data=ua_ut_seasons,
             x='year',
             y='value',
             hue='school_record',
             style='school_record',
             palette=school_colors,
             markers=True, # Mark each season
             dashes=False # Solid lines for both schools
             )

plt.title('Alabama and Tennessee Football Season Records, 1996-2025')
plt.xlabel('Season')
plt.ylabel('Share of Games Won')
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0)) # Show 0.8 as 80%
plt.legend(title='School')

plt.show()

## Scatter plots

Scatter plots show the relation between two numeric variables, with one point per observation. We can add more dimensions with `hue` (color), `size`, and `style` (marker shape), but each one makes the chart harder to read, so add them sparingly. We used `sns.scatterplot()` in [Statistical Concepts](../03_data_analytics_statistics/statistical_concepts.ipynb) to look at price against carat. Let's draw it again with the default settings.

In [ ]:
sns.scatterplot(data=diamonds_small,
                x='carat',
                y='price',
                hue='clarity'
                )

plt.title('Diamond Price by Carat and Clarity')
plt.xlabel('Carat')
plt.ylabel('Price (USD)')

plt.show()

Two things are wonky with this chart.

First, clarity is an **ordinal** variable. The categories run from I1 (worst) to IF (best), but the default palette assigns unrelated colors, so the order is lost. For an ordered variable we want a **sequential** palette that runs from dark to light (or light to dark). `seaborn` has several; `magma` is one we used before.

Second, the chart suffers from **overplotting**. There are about 50,000 diamonds here, and the points sit on top of each other, so we cannot see where the data is dense and where it is sparse. There are two common fixes. One is to plot a random sample of the data. Here we simply "wrap" the dataset in the `sample()` method we used in [Working with Data](../04_working_data), fixing `random_state` so the plot is reproducible.

In [ ]:
clarity_palette = sns.color_palette('magma',
                                    n_colors=len(diamonds_small['clarity'].unique())
                                    ) # One color per clarity level, dark to light

sns.scatterplot(data=diamonds_small.sample(n=1000, random_state=42), # Plot a random sample of 1,000 diamonds
                x='carat',
                y='price',
                hue='clarity',
                palette=clarity_palette
                )

plt.title('Diamond Price by Carat and Clarity (1,000 Sampled Diamonds)')
plt.xlabel('Carat')
plt.ylabel('Price (USD)')

plt.show()

That is better. The other fix is to make the points partially transparent with the `alpha` argument, which runs from 0 (invisible) to 1 (solid). Where many points overlap, the color builds up, so dense regions look darker. This keeps every observation on the chart.

In [ ]:
sns.scatterplot(data=diamonds_small,
                x='carat',
                y='price',
                hue='clarity',
                palette=clarity_palette,
                alpha=0.2 # Make each point 20% opaque
                )

plt.title('Diamond Price by Carat and Clarity')
plt.xlabel('Carat')
plt.ylabel('Price (USD)')

plt.show()

Not bad! Notice that the variation in price gets larger as the diamonds get larger, and that the relation curves somewhat upward rather than following a straight line. A log transformation of both axes straightens out this kind of pattern. We will come back to why when we get to regression. In [Statistical Concepts](../03_data_analytics_statistics/statistical_concepts.ipynb) we took the log of the data itself. Here we take a different route and change the scale of the axes instead, which keeps the tick labels in dollars and carats.

In [ ]:
sns.scatterplot(data=diamonds_small,
                x='carat',
                y='price',
                hue='clarity',
                palette=clarity_palette,
                alpha=0.2
                )

plt.xscale('log') # Log scale on the x-axis
plt.yscale('log') # Log scale on the y-axis

plt.title('Diamond Price by Carat and Clarity (Log Scales)')
plt.xlabel('Carat')
plt.ylabel('Price (USD)')

plt.show()

Now the relation is roughly linear, and the effect of clarity is easy to see: at any given carat, the darker (higher clarity) points sit above the lighter ones.

## Axis scaling

The scale of the x or y axis can dramatically change the appearance of a chart, and the conclusions people draw from it. Below, we plot Alabama's season records from 2011 to 2025 twice, with the same data and different y-axes.

First, we let `seaborn` pick the y-axis limits. It "zooms in" so the data fills the chart.

In [ ]:
ua_recent = ua_ut_seasons[
    (ua_ut_seasons['school_record'] == 'Alabama') & (ua_ut_seasons['year'] > 2010)
] # Alabama seasons from 2011 onward

sns.lineplot(data=ua_recent,
             x='year',
             y='value',
             color='#9E1B32',
             marker='o'
             )

plt.title('Alabama Football Season Records, 2011-2025 (Zoomed In)')
plt.xlabel('Season')
plt.ylabel('Share of Games Won')

plt.show()

This looks a bit like a roller coaster. Now let's set the y-axis to a wider range of possibilities with `plt.ylim()`.

In [ ]:
sns.lineplot(data=ua_recent,
             x='year',
             y='value',
             color='#9E1B32',
             marker='o'
             )

plt.ylim(0, 1.1) # Show the full range of possible values
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))

plt.title('Alabama Football Season Records, 2011-2025 (Zoomed Out)')
plt.xlabel('Season')
plt.ylabel('Share of Games Won')

plt.show()

Same data, very different impression. The zoomed-in chart emphasizes year-to-year variation. The zoomed-out chart emphasizes that Alabama has won at least two thirds of its games in every one of these seasons. Neither is wrong per se, but each answers a different question, so pick the scale that matches the question you are actually asking... and never pick it to exaggerate. Whenever you look at someone else's chart, check the axes first!

## Figure-level functions: `relplot`, `catplot`, and `displot`

Every function we have used so far draws a single chart on a single set of axes. `seaborn` calls these **axes-level** functions. They are organized into three families, and each family has a **figure-level** function that wraps the whole family:

| Family | What it shows | Axes-level functions | Figure-level function |
|--------|---------------|----------------------|-----------------------|
| Relational | How two numeric variables relate | `scatterplot`, `lineplot` | `relplot` |
| Distributions | The distribution of a numeric variable | `histplot`, `kdeplot`, `ecdfplot` | `displot` |
| Categorical | A numeric variable across categories | `barplot`, `countplot`, `boxplot`, `violinplot`, `stripplot` | `catplot` |

A figure-level function takes a `kind` argument that says which chart in the family to draw. `sns.relplot(kind='line')` draws the same thing as `sns.lineplot()`, and `sns.catplot(kind='box')` draws the same thing as `sns.boxplot()`. So why bother? Two reasons:

1. **Faceting.** Figure-level functions can split the data into a grid of subplots with `row=` and `col=`, one panel per group, all sharing the same axes. This is the main reason to use them. Comparing groups side by side on identical axes is often clearer than piling everything onto one chart.
2. **Switching kinds.** Changing `kind='box'` to `kind='violin'` redraws the whole grid with a different chart type, which is handy when exploring.

There are a few differences to be aware of, because the figure-level function creates and owns the whole figure:

- **Size** is set with `height=` (the height of each panel in inches) and `aspect=` (width divided by height). `plt.figure(figsize=...)` will not work.
- **Labels** for the whole grid are set through the object the function returns. Store it in a variable (we use `grid`) and call `grid.set_axis_labels()`, `grid.set_titles()`, and `grid.figure.suptitle()`. Calling `plt.title()` only labels the last panel. We did this in the skewness plot in [Statistical Concepts](../03_data_analytics_statistics/statistical_concepts.ipynb).
- **The legend** is placed outside the plot area by default.

Our rule of thumb: reach for the axes-level function when you want one chart, and for the figure-level function when you want facets. Let's redraw the football chart with one panel per school. Because each panel has a title, we do not need the legend.

In [ ]:
grid = sns.relplot(data=ua_ut_seasons,
                   x='year',
                   y='value',
                   hue='school_record',
                   kind='line', # Same chart as sns.lineplot()
                   row='school_record', # One panel per school, stacked vertically
                   palette=school_colors,
                   height=2.5, # Height of each panel in inches
                   aspect=3, # Each panel is three times as wide as it is tall
                   legend=False # The panel titles already name the school
                   )

grid.set_axis_labels('Season', 'Share of Games Won') # Label the axes on every panel
grid.set_titles('{row_name}') # Use the school name as each panel title
grid.figure.suptitle('Football Season Records by School, 1996-2025', y=1.03) # Add an overall title

plt.show()

# Best practices: exploring, presenting, and some dos and don'ts

Now that we know how to draw the charts, let's talk about what they are for. As the diagram shows, the data scientist sits between the data and the reader. Facing the data, we make charts to **explore** and analyze. Facing the reader, we make charts to **explain**. These are different jobs, and they call for different charts.

## Exploring: fast and disposable

When exploring, *you* are the audience. You already know the data or are otherwise exploring it. Default labels, default colors, and default sizes are fine. Cosmetics and formatting are not the priority.

Here is what an exploratory look at fares by borough might have looked like before we built anything else with the taxi data.

In [ ]:
sns.boxplot(data=taxis, 
            x='pickup_borough', 
            y='fare'
            )

plt.show()

No title, no labels, no formatting, and that is fine. It took one line to see that fares are right-skewed in every borough and that Queens stands out. That is the point of an exploratory chart: it answers your question or tells you where to look next.

## Presenting: one chart, one message

When presenting, the audience is someone else: a manager, a client, a partner, an audit committee. Your audience is unlikely to understand the data and the analysis as well as you do, so it is your responsibility to present it faithfully, such that they come away with an accurate understanding of what can and cannot be learned from it.

There is no formula for a good presentation chart, but there are habits that almost always help:

- **One message per chart.** If a chart makes three points, make three charts.
- **Label the axes with units.** Dollars, percentages, billions, years. Format the tick labels the way the audience thinks about the numbers (commas, dollar signs, percent signs).
- **Order the categories deliberately.** Sort bars by value unless the categories have a natural order of their own (months, credit ratings, class).
- **Make color mean something.** Use a sequential palette for ordered variables. Use the same color for the same category everywhere (Alabama is always crimson). Rarely use more than five or six colors.
- **Keep the finding intact.** The polished chart must not lose or distort what the exploratory chart showed.
- **Simplicity.** Less is more. Every element on the chart should be there for a reason.

### The hallway test

The best check is also the simplest. Before you show a chart to your audience, show it to someone who was not part of the analysis. Say nothing. Ask them what the chart shows. If they need you to explain it extensively, the chart is not finished. **A good presentation chart should need little additional context to interpret.** Assume it will be screenshotted, pasted into an email, and forwarded to someone who never saw your presentation.

## Dos and don'ts

A small number of mistakes account for most bad charts. Each "don't" below is paired with the fix.

### Don't overplot

We covered this in the scatter plot section. When thousands of points sit on top of each other, the chart becomes a solid blob and hides where the data actually is. Here is the full diamonds scatter plot again, with nothing done about it.

In [ ]:
sns.scatterplot(data=diamonds_small,
                x='carat',
                y='price'
                )

plt.title("Don't: Overplotting (Every Diamond, Solid Points)")
plt.xlabel('Carat')
plt.ylabel('Price (USD)')

plt.show()

The fixes are to plot a **sample** or to use **transparency** (`alpha`). We used both earlier, here, we combine: 

In [ ]:
clarity_palette = sns.color_palette('magma',
                                    n_colors=len(diamonds_small['clarity'].unique())
                                    ) # One color per clarity level, dark to light

sns.scatterplot(data=diamonds_small.sample(n=5000, random_state=42), # Plot a random sample of 5,000 diamonds
                x='carat',
                y='price',
                hue='clarity',
                alpha=0.2, # Make each point 20% opaque
                palette=clarity_palette
                )

plt.title('Do: Use a Random Sample of Diamonds and Add Transparency')
plt.xlabel('Carat')
plt.ylabel('Price (USD)')

plt.show()

### Don't use the wrong chart for the task

Each chart type answers a particular kind of question. Match the chart to the question.

| Question | Chart |
|----------|-------|
| How does a value compare across categories? | Bar chart |
| What does the distribution of a variable look like? | Histogram (or boxplots to compare groups) |
| How does a value change over time? | Line chart |
| How are two numeric variables related? | Scatter plot |

Two mismatches come up constantly. The first is a **line chart for categories**. A line implies that the value moves continuously from one point to the next, which means nothing when the categories have no order.

In [ ]:
fare_by_borough = (
    taxis
    .groupby('pickup_borough')
    .agg(mean_fare=('fare', 'mean')) # Average fare in each borough
    .sort_values('mean_fare', ascending=False) # Largest first
    .reset_index()
)

fare_by_borough

In [ ]:
sns.lineplot(data=fare_by_borough,
             x='pickup_borough',
             y='mean_fare',
             marker='o'
             )

plt.title("Don't: A Line Chart for Unordered Categories")
plt.xlabel('Pickup Borough')
plt.ylabel('Average Fare (USD)')

plt.show()

The line suggests the fare "falls" from Queens to Manhattan, but the boroughs are in this order only because we sorted them. A bar chart is the right chart for comparing a value across categories.

In [ ]:
sns.barplot(data=fare_by_borough,
            x='mean_fare',
            y='pickup_borough'
            )

plt.title('Do: A Bar Chart for Categories')
plt.xlabel('Average Fare (USD)')
plt.ylabel('Pickup Borough')
plt.gca().xaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))

plt.show()

The second mismatch is a **pie chart with many slices**. People are good at comparing lengths and bad at comparing angles and areas. Try to rank Germany, Japan, and India from the pie below. (`seaborn` does not include a pie chart at all, so we drop down to `matplotlib` for this one.)

In [ ]:
plt.figure(figsize=(6, 6))

plt.pie(gdp_top10['2022'],
        labels=gdp_top10['Country']
        ) # Pie charts are matplotlib only

plt.title("Don't: A Pie Chart with Many Slices (GDP, 2022)")

plt.show()

Now the same data as a horizontal bar chart, which we drew earlier in the notebook. The ranking is immediate, and so is the size of the gap between the United States and everyone else.

In [ ]:
plt.figure(figsize=(8, 4))

sns.barplot(data=gdp_top10,
            x='2022',
            y='Country'
            )

plt.title('Do: A Bar Chart for Comparing GDP (2022)')
plt.xlabel('GDP (Billions of Current USD)')
plt.ylabel('')
plt.gca().xaxis.set_major_formatter(mtick.StrMethodFormatter('{x:,.0f}'))

plt.show()

### Don't use two y-axes

A chart with two y-axes plots two series with different units on the same axes, one scale on the left and one on the right. The problem is that the two scales are arbitrary. By stretching either axis you can make the lines cross wherever you like, or not at all, and the reader has no way to know that. Here is an example using the number of taxi rides and the average fare by hour of the day. We create the hour with the `.dt.hour` accessor, the same family of date tools we used in [Working with Data](../04_working_data).

In [ ]:
taxis['pickup_hour'] = taxis['pickup'].dt.hour # Hour of the day (0 to 23) of each pickup

rides_by_hour = (
    taxis
    .groupby('pickup_hour')
    .agg(
        rides=('fare', 'count'),
        mean_fare=('fare', 'mean')
    )
    .reset_index()
    .rename(columns={'rides': 'Number of Rides',
                     'mean_fare': 'Average Fare (USD)'}) # Readable panel titles
)

rides_by_hour.head()

In [ ]:
fig, ax_left = plt.subplots(figsize=(8, 4))

sns.lineplot(data=rides_by_hour,
             x='pickup_hour',
             y='Number of Rides',
             color='steelblue',
             ax=ax_left
             )

ax_right = ax_left.twinx() # A second y-axis that shares the x-axis

sns.lineplot(data=rides_by_hour,
             x='pickup_hour',
             y='Average Fare (USD)',
             color='darkorange',
             ax=ax_right
             )

ax_right.grid(False) # Two sets of grid lines would be even worse
ax_left.set_ylabel('Number of Rides', color='steelblue')
ax_right.set_ylabel('Average Fare (USD)', color='darkorange')
ax_left.set_xlabel('Pickup Hour')
ax_left.set_title("Don't: Two Y-Axes on One Chart")

plt.show()

Where the lines cross means little, yet every reader's eye goes straight to it. The fix is to give each measure its own panel, stacked so they share the x-axis. This is exactly what faceting is for. We reshape the summary from wide to long with `melt()`, as in [Working with Data](../04_working_data), so that `relplot` can draw one row per measure. Because the two measures have different units, we let each panel pick its own y-axis with `facet_kws={'sharey': False}`.

In [ ]:
rides_by_hour_long = rides_by_hour.melt(
    id_vars='pickup_hour',
    var_name='measure',
    value_name='value'
) # One row per hour and measure

grid = sns.relplot(data=rides_by_hour_long,
                   x='pickup_hour',
                   y='value',
                   kind='line',
                   row='measure', # One panel per measure
                   facet_kws={'sharey': False}, # Each panel gets its own y-axis
                   height=2.5,
                   aspect=3
                   )

grid.set_axis_labels('Pickup Hour', '')
grid.set_titles('{row_name}')
grid.figure.suptitle('Do: One Panel per Measure, Sharing the X-Axis', y=1.03)

plt.show()

Now each series can be read on its own scale, and the shared x-axis still lets the reader line up the hours.